### README 

In [61]:
# This script processes a dataset to filter and save a specific level,
# and includes preprocessing steps to engineer student goals, impasses,
# and tag whether students quit or win the level.

### Import Packages

In [62]:
import pandas as pd
import numpy as np

### Define File Paths

In [63]:
brick_wall = "/Users/baselhussein/Projects/impossible_goals/data/brick_wall.csv"
output_win = "/Users/baselhussein/Projects/impossible_goals/data/conditions/brick_wall_win.csv"
output_quit = "/Users/baselhussein/Projects/impossible_goals/data/conditions/brick_wall_quit.csv"

### Preprocessing 

In [64]:
df_import = pd.read_csv(brick_wall)

In [65]:
df_import.head()

,ID,Session_timestamp,Level_timestamp,Event_type,Event,Level,Details,filename,epoch_time,object_orientation,x_coordinate,y_coordinate,objects,Session_number,Level_progression,Ruleset,Levels_complete,LevelTime,SessionTime,drop
0,100,0:50:14:633,0:0:0:000,event,start,93level:brick wall,NaN,100_0.txt,1.643223e+12,NaN,NaN,NaN,NaN,NaN,14.0,NaN,"['0level:baba is you', '1level:where do i go?'...",NaN,NaN,NaN
1,100,0:50:14:633,0:0:0:000,event,init_object,93level:brick wall,algae:1:2,100_0.txt,1.643223e+12,NaN,1.0,2.0,algae,NaN,NaN,NaN,"['0level:baba is you', '1level:where do i go?'...",NaN,NaN,NaN
2,100,0:50:14:633,0:0:0:000,event,init_object,93level:brick wall,algae:2:1,100_0.txt,1.643223e+12,NaN,2.0,1.0,algae,NaN,NaN,NaN,"['0level:baba is you', '1level:where do i go?'...",NaN,NaN,NaN
3,100,0:50:14:633,0:0:0:000,event,init_object,93level:brick wall,baba:2:5,100_0.txt,1.643223e+12,NaN,2.0,5.0,baba,NaN,NaN,NaN,"['0level:baba is you', '1level:where do i go?'...",NaN,NaN,NaN
4,100,0:50:14:633,0:0:0:000,event,init_object,93level:brick wall,algae:3:8,100_0.txt,1.643223e+12,NaN,3.0,8.0,algae,NaN,NaN,NaN,"['0level:baba is you', '1level:where do i go?'...",NaN,NaN,NaN


In [66]:
columns_to_drop = ['filename', 'Session_number', 'Levels_complete', 
                   'LevelTime', 'SessionTime', 'drop', 'Level_progression']

In [67]:
df = df_import.drop(columns=columns_to_drop)

#### Tag Student Goals

##### Goal 1: BABA-IS-YOU & FLAG-IS-WIN

In [68]:
# Init column
df['goal_1'] = ''

# Iterate over df 
for index, row in df.iterrows():
    
    # Convert ruleset as str
    ruleset = str(row['Ruleset'])
    
    # Check ruleset 
    if "baba is you " in ruleset and "flag is win " in ruleset:
        df.at[index, 'goal_1'] = 'goal_1'

##### Goal 2: FLAG-IS-YOU

In [69]:
# Init column
df['goal_2'] = ''

# Loop 
for index, row in df.iterrows():
    
    # Update ruleset to str
    ruleset = str(row['Ruleset'])
    
    # Check conditions
    if "flag is you " in ruleset and "flag is win " not in ruleset and "baba is win " not in ruleset:
        df.at[index, 'goal_2'] = 'goal_2'

##### Goal 3: BABA-IS-WIN

In [70]:
# Init column
df['goal_3'] = ''

# Loop 
for index, row in df.iterrows():
    
    # Check conditions for the current row
    if ("baba is win " in str(row['Ruleset'])
        and row['Event'] != 'rule_remove'
        and "baba is you " not in str(row['Ruleset'])
#         and "flag is baba " not in str(row['Ruleset'])
        and "flag is you " not in str(row['Ruleset'])
#         and "flag is you " not in str(df.at[index + 1, 'Ruleset'])
        ): 
        df.at[index, 'goal_3'] = 'goal_3'

##### Goal 4: BABA-IS-WIN & FLAG-IS-YOU

In [71]:
# Init column
df['goal_4'] = ''

# Loop 
for index in df.index:
    
    # Update ruleset to str
    ruleset = str(df.loc[index, 'Ruleset'])
    
    # Check conditions
    if "flag is you " in ruleset and "baba is win " in ruleset:
        df.loc[index, 'goal_4'] = 'goal_4'

##### Goal 5: BABA-IS-YOU-IS-WIN (horizontally)

In [72]:
# Init column 
df['goal_5'] = ''

# Iterate over df rows
tagging = False 

for index, row in df.iterrows():
    if tagging: 
        if (row['Event'] == 'undo' or
            row['objects'] == 'text_win' or 
            row['Event'] == 'rule_remove' or 
            'flag is baba ' in row['Ruleset'] or
            row['Event'] == 'restart'):
            tagging = False
        else: 
            df.at[index, 'goal_5'] = 'goal_5'
        
    else:
        # Check for conditions to start tagging rows
        if row['Event'] == 'update' and row['objects'] == 'text_win':
            
            # Filter rows before the current row
            preceding_rows = df.loc[:index - 1]

            # Find the last occurrences of relevant objects and rules 
            last_baba = preceding_rows.loc[preceding_rows['objects'] == 'text_baba'].iloc[-1]
            last_you = preceding_rows.loc[preceding_rows['objects'] == 'text_you'].iloc[-1]
            last_is = preceding_rows.loc[preceding_rows['objects'] == 'text_is'].iloc[-1]
            last_rule = preceding_rows.loc[preceding_rows['objects'] == 'baba is you '].iloc[-1]

            # Check conditions for goal state
            if (row['Details'][11] 
                == last_baba['Details'][12] 
                == last_you['Details'][11] 
                == last_rule['Details'][2]
                == last_is['Details'][10]
                and int(row['Details'][9]) == int(last_you['Details'][9]) + 2  # Check if WIN and YOU are 2 units away on the x-axis
                and 'baba is you ' in row['Ruleset']
                and 'flag is you ' not in row['Ruleset']
                and 'flag is baba ' not in row['Ruleset']
                and 'flag is win ' not in row['Ruleset']):
                df.at[index, 'goal_5'] = 'goal_5'
                tagging = True # Tag following rows 

##### Goal 6: FLAG-IS-YOU-IS-WIN (horizontally)

In [73]:
# Init column 
df['goal_6'] = ''

# Iterate over DataFrame rows
tagging = False  # Flag to indicate whether to continue tagging rows

for index, row in df.iterrows():
    if tagging:
        # Tag rows until an "undo" event or another instance of "text_win" is encountered
        if row['Event'] == 'undo' or row['objects'] == 'text_win' or row['Event'] == 'rule_remove' or row['Event'] == 'restart':
            tagging = False
        else:
            df.at[index, 'goal_6'] = 'goal_6'
   
    else:
        # Check for conditions to start tagging rows
        if row['Event'] == 'update' and row['objects'] == 'text_win':
            # Filter rows before the current row
            preceding_rows = df.loc[:index - 1]

            # Find the last occurrences of relevant objects and rules 
            last_flag = preceding_rows.loc[preceding_rows['objects'] == 'text_flag'].iloc[-1]
            last_you = preceding_rows.loc[preceding_rows['objects'] == 'text_you'].iloc[-1]
            #last_is = preceding_rows.loc[preceding_rows['objects'] == 'text_is'].iloc[-1]
            last_rule = preceding_rows.loc[preceding_rows['objects'] == 'flag is you '].iloc[-1]

            # Check conditions for goal state
            if (row['Details'][11] 
                == last_flag['Details'][12] 
                == last_you['Details'][11] 
                == last_rule['Details'][2]
                # == last_is["Details"][10]
                and int(row['Details'][9]) == int(last_you['Details'][9]) + 2  # Check if WIN and YOU are 2 units away on the x-axis
                and 'flag is you ' in row['Ruleset']):
                df.at[index, 'goal_6'] = 'goal_6'
                tagging = True # Tag following rows

##### Goal 7: BABA-IS-FLAG (horizontally)

In [74]:
# Init column
df['goal_7'] = ''

# Loop 
for index in df.index:
    
    # Update ruleset to str
    ruleset = str(df.loc[index, 'Ruleset'])
    
    # Check conditions
    if "baba is flag " in ruleset and "baba is you " not in ruleset:
        df.loc[index, 'goal_7'] = 'goal_7'

##### Goal 8: BABA-IS-FLAG-IS-YOU

In [75]:
# Init column
df['goal_8'] = ''

# Loop 
for index in df.index:
    
    # Update ruleset to str
    ruleset = str(df.loc[index, 'Ruleset'])
    
    # Check conditions
    if "baba is flag " in ruleset and "flag is you " in ruleset:
        df.loc[index, 'goal_8'] = 'goal_8'

##### Goal 9: BABA-IS-YOU + YOU-IS-WIN (vertical)

In [76]:
# Init column 
df['goal_9'] = ''

# Iterate over DataFrame rows
tagging = False  # Flag to indicate whether to continue tagging rows

for index, row in df.iterrows():
    if tagging:

        # Tag rows until an "undo" event or another instance of "text_win" is encountered
        if row['Event'] == 'undo' or row['objects'] == 'text_win' or row['Event'] == 'rule_remove' or row['Event'] == 'restart':
            tagging = False
        else:
            df.at[index, 'goal_9'] = 'goal_9'
   
    else:
        # Check for conditions to start tagging rows
        if row['Event'] == 'update' and row['objects'] == 'text_win':
            # Filter rows before the current row
            preceding_rows = df.loc[:index - 1]

            # Find the last occurrences of relevant objects and rules 
            last_you = preceding_rows.loc[preceding_rows['objects'] == 'text_you'].iloc[-1]
            last_is = preceding_rows.loc[preceding_rows['objects'] == 'text_is'].iloc[-1]
            last_rule = preceding_rows.loc[preceding_rows['objects'] == 'baba is you '].iloc[-1]

            # Check conditions for goal state
            if (row['Details'][9] 
                == last_is['Details'][8] 
                == last_you['Details'][9] 
                != last_rule['Details'][0]
                
                # y-axis
                and int(row['Details'][11]) == int(last_is['Details'][10]) + 1 
                and int(row['Details'][11]) == int(last_you['Details'][11]) + 2
    
                
                and int(row['Details'][11]) == int(last_you['Details'][11]) + 2  # Check if WIN and YOU are 2 units away on the y-axis
                and int(last_you['Details'][11]) == int(last_is['Details'][10]) - 1
                and 'baba is you ' in row['Ruleset']):
                df.at[index, 'goal_9'] = 'goal_9'
                tagging = True # Tag following rows

##### Goal 10: FLAG-IS-BABA-IS-YOU

In [77]:
# Init column
df['goal_10'] = ''

# Loop 
for index in df.index:
    
    # Update ruleset to str
    ruleset = str(df.loc[index, 'Ruleset'])
    
    # Check conditions
    if "flag is baba " in ruleset and "baba is you " in ruleset:
        df.loc[index, 'goal_10'] = 'goal_10'

##### Goal 12: BABA-IS-YOU & BABA-IS-FLAG

In [78]:
# Init column
df['goal_12'] = ''

# Loop 
for index in df.index:
    
    # Update ruleset to str
    ruleset = str(df.loc[index, 'Ruleset'])
    
    # Check conditions
    if "baba is flag " in ruleset and "baba is you " in ruleset:
        df.loc[index, 'goal_12'] = 'goal_12'

##### Goal 14: BABA-IS-YOU & FLAG-IS-BABA

In [79]:
# Init column
df['goal_14'] = ''

# Loop 
for index in df.index:
    
    # Update ruleset to str
    ruleset = str(df.loc[index, 'Ruleset'])
    
    # Check conditions
    if "flag is baba " in ruleset and "flag is you " in ruleset:
        df.loc[index, 'goal_14'] = 'goal_14'

##### Goal 16: Solution A, BABA IS YOU & BABA IS WIN

In [80]:
# Init column
df['goal_16'] = ''

# Loop 
for index in df.index:
    
    # Update ruleset to str
    ruleset = str(df.loc[index, 'Ruleset'])
    
    # Check conditions
    if "baba is win " in ruleset and "baba is you "  in ruleset:
        df.loc[index, 'goal_16'] = 'goal_16'

##### Goal 17: Solution B, FLAG IS YOU & FLAG IS WIN

In [81]:
# Init column
df['goal_17'] = ''

# Loop 
for index in df.index:
    
    # Update ruleset to str
    ruleset = str(df.loc[index, 'Ruleset'])
    
    # Check conditions
    if "flag is win " in ruleset and "flag is you "  in ruleset:
        df.loc[index, 'goal_17'] = 'goal_17'

##### Goal 0: Exploratory

In [82]:
df['goal_0'] = ''

In [83]:
goal_columns = ['goal_0',
                'goal_1', 
                'goal_2', 
                'goal_3', 
                'goal_4', 
                'goal_5', 
                'goal_6', 
                'goal_7', 
                'goal_8', 
                'goal_9', 
                'goal_10', 
                #'goal_11', 
                'goal_12',
                #'goal_13',
                'goal_14',
                #'goal_15',
                'goal_16',
                'goal_17'] 

In [84]:
df[goal_columns] = df[goal_columns].replace('', np.nan)

In [85]:
# Check if all entries across the specified columns are NaN
all_none = df[goal_columns].isna().all(axis=1)

# Add 'no_goal' to the "no_goal" column where all entries across the specified columns are NaN
df.loc[all_none, 'goal_0'] = 'goal_0'

#### Validate Goals

In [86]:
# Check that there are no overlapping goals in data

In [87]:
# Count non-NaN values across the selected columns for each row
non_nan_counts = df[goal_columns].count(axis=1)

# Check if there are any rows where there is more than one non-NaN value
rows_with_multiple_non_nan = non_nan_counts[non_nan_counts > 1].index

print("Row numbers with more than one non-NaN value across the selected columns:")
print(rows_with_multiple_non_nan)

Row numbers with more than one non-NaN value across the selected columns:
Index([], dtype='int64')


#### Add Impasses

##### Impasse 1: Baba Hits Walls 

In [88]:
# Create coordinates column 

df['coordinates'] = ('(' + df['x_coordinate'].astype("string") + ',' + 
                            df['y_coordinate'].astype("string") + ')') 

In [89]:
# Set NaN coordinates to 'None'

df['coordinates'] = df['coordinates'].fillna('None')

In [90]:
# List of coordinates to check 

baba_at_wall = [# bottom wall
                '(10.0,8.0)', '(11.0,8.0)', '(12.0,8.0)', '(13.0,8.0)', '(14.0,8.0)', 
    
                # top wall
                '(10.0,2.0)', '(11.0,2.0)', '(12.0,2.0)', '(13.0,2.0)', '(14.0,2.0)',
    
                # left wall
                '(9.0,3.0)', '(9.0,4.0)', '(9.0,5.0)', '(9.0,6.0)', '(9.0,7.0)']
    
                       
# Init column 
df['impasse_1'] = ''

# Function to check conditions
for i in range(len(df)): 
    if ("baba is you" in str(df.loc[df.index[i], 'Ruleset']) and
        df.loc[df.index[i], 'coordinates'] in baba_at_wall and 
        df.loc[df.index[i+1], 'Event_type'] == "input" and 
        df.loc[df.index[i+2], 'Event_type'] == "input"):
        
        # Tag impasse moment
        df.loc[df.index[i], 'impasse_1'] = 'impasse_1'
        
        # Tag following input rows until no longer input
        j = i + 1
        while j < len(df) and df.loc[df.index[j], 'Event_type'] == "input" and df.loc[df.index[j], 'Event'] != "undo":
            df.loc[df.index[j], 'impasse_1'] = 'impasse_1'
            j += 1

##### Impasse 2: Baba Hits Rule

In [91]:
# List of coordinates to check 
baba_at_rule = ['(15.0,2.0)', '(15.0,8.0)']
    
                       
# Init column 
df['impasse_2'] = ''

# Function to check conditions
for i in range(len(df)): 
    if (df.loc[df.index[i], 'objects'] == 'baba' and 
        df.loc[df.index[i], 'coordinates'] in baba_at_rule and 
        df.loc[df.index[i+1], 'Event_type'] == "input" and 
        df.loc[df.index[i+2], 'Event_type'] == "input"
        
         ):
        
            # tag impasse moment
            df.loc[df.index[i], 'impasse_2'] = 'impasse_2' 

##### Impasse 3: Flag Hits Walls

In [92]:
# List of coordinates to check 
flag_at_wall = ['(11.0,4.0)', '(12.0,4.0)',
                '(11.0,5.0)',
                '(11.0,6.0)', '(12.0,6.0)']
    
                       
# Init column 
df['impasse_3'] = ''

# Function to check conditions
for i in range(len(df)): 
    if (df.loc[df.index[i], 'objects'] == 'flag' and 
        df.loc[df.index[i], 'coordinates'] in flag_at_wall and 
        df.loc[df.index[i+1], 'Event_type'] == "input" and 
        df.loc[df.index[i+2], 'Event_type'] == "input"
        
         ):
        
            # tag impasse moment
            df.loc[df.index[i], 'impasse_3'] = 'impasse_3' 

##### Impasse 4: Flag Hits Rules 

In [93]:
# List of coordinates to check 
flag_at_rule = ['(13.0,4.0)', '(13.0,5.0)', '(13.0,6.0)']
    
                       
# Init column 
df['impasse_4'] = ''

# Function to check conditions
for i in range(len(df)): 
    if (df.loc[df.index[i], 'objects'] == 'flag' and 
        df.loc[df.index[i], 'coordinates'] in flag_at_rule and 
        df.loc[df.index[i+1], 'Event_type'] == "input" and 
        df.loc[df.index[i+2], 'Event_type'] == "input"
        
         ):
        
            # tag impasse moment
            df.loc[df.index[i], 'impasse_4'] = 'impasse_4' 

##### Impasse 5: no_you

In [94]:
# Init column
df['impasse_5'] = ''

# Function to check coordinates 
for i in range(len(df)): 
    if (df.loc[df.index[i], 'Event'] == 'no_you'):
        
        # Tag impasse moment 
        df.loc[df.index[i], 'impasse_5'] = 'impasse_5' 

##### Impasse 6: Syntax 

In [95]:
# List of columns to check

syntax_impasse_columns = ['goal_5', 
                          'goal_6', 
                          'goal_7', 
                          'goal_8', 
                          'goal_9', 
                          'goal_10', 
                          #'goal_11', 
                          'goal_12',
                          #'goal_13',
                          'goal_14'
                          #'goal_15'
                         ] 

In [96]:
df['impasse_6'] = ''


# Function to update 'impasse_6' if any value is present in the specified columns
def update_impasse_6(row, syntax_impasse_columns):
    if any(pd.notna(row[col]) for col in syntax_impasse_columns):
        return 'impasse_6'
    return row['impasse_6']


# Apply the function to each row
df['impasse_6'] = df.apply(update_impasse_6, axis=1, syntax_impasse_columns=syntax_impasse_columns)

#### Tag if Player 'Wins' or 'Quits'

In [97]:
# Init column 
df['win_or_quit'] = ''

# Loop through and tag 'win' moment or the last row as 'quit'
for id in df['ID'].unique():
    
    # Get the subset of data for the current ID
    id_data = df[df['ID'] == id]
    
    # Find 'win' moment, if any
    win_indices = id_data[id_data['Event'] == 'win'].index
    
    if not win_indices.empty:
        # Tag 'win' moment
        df.loc[win_indices, 'win_or_quit'] = 'win'
        
    else:
        # Tag the last row as 'quit'
        last_row_index = id_data.index[-1]
        df.loc[last_row_index, 'win_or_quit'] = 'quit'

#### Data Cleaning 

##### Drop rows & columns 

In [98]:
df_drop_rows = df[~df['Event'].isin(['start', 'init_rule', 'init_object'])]

In [99]:
df_clean = df_drop_rows.drop(columns=['x_coordinate', 'y_coordinate', 
                                      'object_orientation', 'objects', 
                                       'coordinates', 'Level'])

In [100]:
df_clean = df_clean.replace(np.nan, '')

In [101]:
# df_clean = df_clean.replace(np.nan, "None")

In [102]:
df_clean.head()

,ID,Session_timestamp,Level_timestamp,Event_type,Event,Details,epoch_time,Ruleset,goal_1,goal_2,...,goal_16,goal_17,goal_0,impasse_1,impasse_2,impasse_3,impasse_4,impasse_5,impasse_6,win_or_quit
70,100,0:50:17:800,0:0:3:166,change,update,baba:3:5:0,1.643223e+12,"['baba is you ', 'flag is win ', 'wall is stop ']",goal_1,,...,,,,,,,,,,
71,100,0:50:17:800,0:0:3:166,input,right,,1.643223e+12,"['baba is you ', 'flag is win ', 'wall is stop ']",goal_1,,...,,,,,,,,,,
72,100,0:50:18:050,0:0:3:416,change,update,baba:4:5:0,1.643223e+12,"['baba is you ', 'flag is win ', 'wall is stop ']",goal_1,,...,,,,,,,,,,
73,100,0:50:18:050,0:0:3:416,input,right,,1.643223e+12,"['baba is you ', 'flag is win ', 'wall is stop ']",goal_1,,...,,,,,,,,,,
74,100,0:50:18:216,0:0:3:583,change,update,baba:5:5:0,1.643223e+12,"['baba is you ', 'flag is win ', 'wall is stop ']",goal_1,,...,,,,,,,,,,


##### Combine Goal Columns 

In [103]:
df_clean['goals'] = df_clean[goal_columns].apply(lambda row: ' '.join(row.values.astype(str)), axis=1)

In [104]:
df_goals_clean = df_clean.drop(columns=goal_columns)

##### Combine Impasse Columns

In [105]:
impasse_columns = ['impasse_1', 'impasse_2', 'impasse_3', 
                   'impasse_4', 'impasse_5', 'impasse_6']

In [106]:
df_goals_clean['impasses'] = df_goals_clean[impasse_columns].apply(lambda row: ' '.join(row.values.astype(str)), axis=1)

In [107]:
df_impasses_clean = df_goals_clean.drop(columns=impasse_columns)

In [108]:
df_impasses_clean.head()

,ID,Session_timestamp,Level_timestamp,Event_type,Event,Details,epoch_time,Ruleset,win_or_quit,goals,impasses
70,100,0:50:17:800,0:0:3:166,change,update,baba:3:5:0,1.643223e+12,"['baba is you ', 'flag is win ', 'wall is stop ']",,goal_1,
71,100,0:50:17:800,0:0:3:166,input,right,,1.643223e+12,"['baba is you ', 'flag is win ', 'wall is stop ']",,goal_1,
72,100,0:50:18:050,0:0:3:416,change,update,baba:4:5:0,1.643223e+12,"['baba is you ', 'flag is win ', 'wall is stop ']",,goal_1,
73,100,0:50:18:050,0:0:3:416,input,right,,1.643223e+12,"['baba is you ', 'flag is win ', 'wall is stop ']",,goal_1,
74,100,0:50:18:216,0:0:3:583,change,update,baba:5:5:0,1.643223e+12,"['baba is you ', 'flag is win ', 'wall is stop ']",,goal_1,


##### Clean strings

In [109]:
df_impasses_clean['goals'] = df_impasses_clean['goals'].str.replace(' ', '')
df_impasses_clean['impasses'] = df_impasses_clean['impasses'].str.replace(' ', '')

##### Clean Overlapping impasse_6 Entries 

In [110]:
df_impasses_clean['impasses'] = df_impasses_clean['impasses'].str.replace('impasse_5impasse_6', 'impasse_5')
df_impasses_clean['impasses'] = df_impasses_clean['impasses'].str.replace('impasse_1impasse_6', 'impasse_1')
df_impasses_clean['impasses'] = df_impasses_clean['impasses'].str.replace('impasse_2impasse_6', 'impasse_2')

In [111]:
df_impasses_clean['impasses'] = df_impasses_clean['impasses'].mask(df_impasses_clean['impasses'].eq('impasse_6') & df_impasses_clean['impasses'].shift().eq('impasse_6'))

#### Calculate Time Offset Between Events 

In [112]:
# Calculate the difference within each group and fill NaN values with 0
df_impasses_clean['time_offset'] = df_impasses_clean.groupby('ID')['epoch_time'].diff().fillna(0)

# If you want to convert the NaNs to integers
df_impasses_clean['time_offset'] = df_impasses_clean['time_offset'].astype(int)

In [113]:
df_impasses_clean.loc[df_impasses_clean['time_offset'] > 120000, 'time_offset'] = 120000
df_impasses_clean.loc[df_impasses_clean['time_offset'] < 0, 'time_offset'] = 0 

In [114]:
# df_impasses_clean

### Split Data into Quit and Win Groups

In [115]:
# Group by ID and check if any occurrence of 'win' exists within each group
win_ids = df_impasses_clean.groupby('ID')['win_or_quit'].transform(lambda x: (x == 'win').any())

# Separate into df_win and df_quit based on win_ids
df_win = df_impasses_clean[win_ids]
df_quit = df_impasses_clean[~win_ids]

#### Keep Only First Pass of Win Data 

In [116]:
# Function to drop rows following the 'win' event for each ID
def drop_rows_after_win(df_win):
    result = []
    for id, group in df_win.groupby('ID'):
        win_idx = group[group['Event'] == 'win'].index
        if not win_idx.empty:
            first_win_idx = win_idx[0]
            group = group.loc[:first_win_idx]
        result.append(group)
    return pd.concat(result)

# Apply the function
df_win_first_pass = drop_rows_after_win(df_win)

In [117]:
df_win_first_pass

,ID,Session_timestamp,Level_timestamp,Event_type,Event,Details,epoch_time,Ruleset,win_or_quit,goals,impasses,time_offset
131600,2,0:16:39:250,0:0:5:083,change,update,baba:3:5:0,1.636142e+12,"['baba is you ', 'flag is win ', 'wall is stop ']",,goal_1,,0
131601,2,0:16:39:250,0:0:5:083,input,right,,1.636142e+12,"['baba is you ', 'flag is win ', 'wall is stop ']",,goal_1,,0
131602,2,0:16:39:416,0:0:5:250,change,update,baba:4:5:0,1.636142e+12,"['baba is you ', 'flag is win ', 'wall is stop ']",,goal_1,,166
131603,2,0:16:39:416,0:0:5:250,input,right,,1.636142e+12,"['baba is you ', 'flag is win ', 'wall is stop ']",,goal_1,,0
131604,2,0:16:39:566,0:0:5:400,change,update,baba:5:5:0,1.636142e+12,"['baba is you ', 'flag is win ', 'wall is stop ']",,goal_1,,150
...,...,...,...,...,...,...,...,...,...,...,...,...
129807,258,4:46:6:050,0:2:8:300,change,update,baba:5:7:1,1.655581e+12,"['wall is stop ', 'baba is you ']",,goal_0,,367
129808,258,4:46:6:050,0:2:8:300,change,update,text_win:5:5:1,1.655581e+12,"['wall is stop ', 'baba is you ']",,goal_0,,0
129809,258,4:46:6:050,0:2:8:300,change,update,baba:5:6:1,1.655581e+12,"['wall is stop ', 'baba is you ']",,goal_0,,0
129810,258,4:46:6:050,0:2:8:300,event,rule_add,5:3:baba is win,1.655581e+12,"['wall is stop ', 'baba is you ', 'baba is win ']",,goal_16,,0


### Export & Validate

In [118]:
# df_win_first_pass.to_csv('/Users/baselhussein/Downloads/df_check.csv', index=False)

### Export & Save 

In [119]:
df_win_first_pass.to_csv(output_win, index=False)
df_quit.to_csv(output_quit, index=False)